# Global Pollution Analysis and Energy Recovery using Apriori Algorithm

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('Global_Pollution_Analysis 3.csv')
print(df.shape)
df.head()


## Data Cleaning

In [ ]:

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(exclude='object').columns:
    df[col] = df[col].fillna(df[col].median())

df.info()


## Feature Engineering

In [ ]:

# Encode categorical columns
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Energy Consumption Per Capita
if 'Energy_Consumption (in GWh)' in df.columns and 'Population' in df.columns:
    df['Energy_Consumption_Per_Capita'] = (
        df['Energy_Consumption (in GWh)'] / (df['Population']+1)
    )

# Pollution Severity Categories
if 'Air_Pollution_Index' in df.columns:
    df['Air_Severity'] = pd.qcut(
        df['Air_Pollution_Index'],
        q=3,
        labels=['Low','Medium','High']
    )

if 'Water_Pollution_Index' in df.columns:
    df['Water_Severity'] = pd.qcut(
        df['Water_Pollution_Index'],
        q=3,
        labels=['Low','Medium','High']
    )

df.head()


## Pollution Trend Analysis

In [ ]:

if 'Year' in df.columns:
    yearly = df.groupby('Year').mean(numeric_only=True)

    yearly[['Air_Pollution_Index',
            'Water_Pollution_Index',
            'Soil_Pollution_Index']].plot(figsize=(10,5))

    plt.title('Pollution Trends Over Years')
    plt.show()


## Apriori Algorithm

In [ ]:

basket = pd.get_dummies(
    df[['Air_Severity','Water_Severity']].astype(str)
)

frequent_itemsets = apriori(
    basket,
    min_support=0.10,
    use_colnames=True
)

frequent_itemsets.sort_values(
    by='support',
    ascending=False
).head()


In [ ]:

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.5
)

rules[['antecedents',
       'consequents',
       'support',
       'confidence',
       'lift']].head()


## Association Rule Visualization

In [ ]:

plt.figure(figsize=(8,5))

plt.scatter(
    rules['support'],
    rules['confidence'],
    s=rules['lift']*50
)

plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules')
plt.show()


## Validation Metrics

In [ ]:

print('Average Support:', rules['support'].mean())
print('Average Confidence:', rules['confidence'].mean())
print('Average Lift:', rules['lift'].mean())


## Final Report


### Methodology
1. Cleaned and encoded pollution dataset.
2. Created pollution severity categories.
3. Analyzed yearly pollution trends.
4. Applied Apriori algorithm to identify frequent itemsets.
5. Generated association rules and evaluated them using support, confidence, and lift.

### Key Findings
- Pollution severity categories reveal recurring environmental patterns.
- Association rules identify relationships between air and water pollution levels.
- High-lift rules indicate strong environmental dependencies.

### Actionable Insights
- Countries with high pollution clusters should focus on waste-to-energy recovery programs.
- Monitor pollution indicators jointly rather than independently.
- Use discovered rules for environmental policy planning and energy optimization.
